In [2]:
import pandas as pd
import numpy as np
import scanpy as sc

In [3]:
adata = sc.read("results/gdt_t.h5ad")
adata

AnnData object with n_obs × n_vars = 55011 × 2099
    obs: 'sample_id', 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', '_scvi_batch', '_scvi_labels', 'doublet', 'size_factors', 'coarse_anno', 'fine_anno', 'gd_anno'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_deviant', 'binomial_deviance', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'cosg', 'dendrogram_gd_anno', 'hvg', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'analytic_pearson_residuals', 'log1p_norm', 'raw_counts', 'scran_normalization', 'soupX_counts'

In [4]:
adata.obs["gd_anno"].value_counts()

gd_anno
abt    53447
gdt     1564
Name: count, dtype: int64

In [5]:
pbmc_df = adata.to_df()
pbmc_df.head()

,HES4,ISG15,TNFRSF18,TNFRSF4,MMP23B,NADK,MEGF6,TNFRSF25,TNFRSF9,ERRFI1,...,CD40LG,LDOC1,AFF2,ZNF185,PNMA3,PDZD4,TKTL1,PLXNA3,MPP1,CLIC2
AAACAAGCACATTGTCACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.825543,1.813600,0.0,0.0,...,1.813600,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0
AAACAAGCACCATACTACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,1.494708,0.000000,0.0,0.0,0.00000,1.494708,0.0,1.003942,0.0,0.0
AAACAAGCATTTGTTGACTTTAGG-1,0.0,0.0,1.256215,0.0,0.0,0.608334,0.608334,0.983850,0.0,0.0,...,0.000000,0.983850,0.0,0.0,0.98385,0.000000,0.0,0.000000,0.0,0.0
AAACCAATCAAACCGGACTTTAGG-1,0.0,0.0,0.663697,0.0,0.0,0.000000,0.663697,0.663697,0.0,0.0,...,2.027320,0.663697,0.0,0.0,0.00000,0.663697,0.0,0.000000,0.0,0.0
AAACCAGGTGATTACCACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,1.328549,0.0,0.0,...,1.879637,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0


In [6]:
pbmc_df.shape

(55011, 2099)

In [7]:
pbmc_df["gd_anno"] = adata.obs["gd_anno"]
#pbmc_df["fine_anno"] = adata.obs["fine_anno"]
pbmc_df.head()

,HES4,ISG15,TNFRSF18,TNFRSF4,MMP23B,NADK,MEGF6,TNFRSF25,TNFRSF9,ERRFI1,...,LDOC1,AFF2,ZNF185,PNMA3,PDZD4,TKTL1,PLXNA3,MPP1,CLIC2,gd_anno
AAACAAGCACATTGTCACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.825543,1.813600,0.0,0.0,...,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,abt
AAACAAGCACCATACTACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.00000,1.494708,0.0,1.003942,0.0,0.0,abt
AAACAAGCATTTGTTGACTTTAGG-1,0.0,0.0,1.256215,0.0,0.0,0.608334,0.608334,0.983850,0.0,0.0,...,0.983850,0.0,0.0,0.98385,0.000000,0.0,0.000000,0.0,0.0,abt
AAACCAATCAAACCGGACTTTAGG-1,0.0,0.0,0.663697,0.0,0.0,0.000000,0.663697,0.663697,0.0,0.0,...,0.663697,0.0,0.0,0.00000,0.663697,0.0,0.000000,0.0,0.0,abt
AAACCAGGTGATTACCACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,1.328549,0.0,0.0,...,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,abt


#### 将pbmc_df分成train和test两个df

In [8]:
pbmc_df.head()

,HES4,ISG15,TNFRSF18,TNFRSF4,MMP23B,NADK,MEGF6,TNFRSF25,TNFRSF9,ERRFI1,...,LDOC1,AFF2,ZNF185,PNMA3,PDZD4,TKTL1,PLXNA3,MPP1,CLIC2,gd_anno
AAACAAGCACATTGTCACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.825543,1.813600,0.0,0.0,...,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,abt
AAACAAGCACCATACTACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.00000,1.494708,0.0,1.003942,0.0,0.0,abt
AAACAAGCATTTGTTGACTTTAGG-1,0.0,0.0,1.256215,0.0,0.0,0.608334,0.608334,0.983850,0.0,0.0,...,0.983850,0.0,0.0,0.98385,0.000000,0.0,0.000000,0.0,0.0,abt
AAACCAATCAAACCGGACTTTAGG-1,0.0,0.0,0.663697,0.0,0.0,0.000000,0.663697,0.663697,0.0,0.0,...,0.663697,0.0,0.0,0.00000,0.663697,0.0,0.000000,0.0,0.0,abt
AAACCAGGTGATTACCACTTTAGG-1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,1.328549,0.0,0.0,...,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,abt


In [10]:
pbmc_df.gd_anno.value_counts()

gd_anno
abt    53447
gdt     1564
Name: count, dtype: int64

In [12]:
import pandas as pd  

# 创建一个空的训练集DataFrame  
pbmc_coarse_train_df = pd.DataFrame()  

# 对每种细胞类型取样150个  
for cell_type in pbmc_df["gd_anno"].unique():  
    # 获取当前细胞类型的所有数据  
    cell_df = pbmc_df[pbmc_df["gd_anno"] == cell_type]  
    
    # 对于每种类型，取150个样本或全部（如果少于150个）  
    sample_size = min(1000, len(cell_df))  
    sampled_df = cell_df.sample(n=sample_size, random_state=42)  
    
    # 将样本添加到训练集  
    pbmc_coarse_train_df = pd.concat([pbmc_coarse_train_df, sampled_df])  

# 剩余的样本作为测试集  
pbmc_coarse_test_df = pbmc_df.loc[~pbmc_df.index.isin(pbmc_coarse_train_df.index)]  

# 验证结果  
print("训练集中每种细胞类型的数量:")  
print(pbmc_coarse_train_df.gd_anno.value_counts())  
print("\n测试集中每种细胞类型的数量:")  
print(pbmc_coarse_test_df.gd_anno.value_counts())  

训练集中每种细胞类型的数量:
gd_anno
abt    1000
gdt    1000
Name: count, dtype: int64

测试集中每种细胞类型的数量:
gd_anno
abt    52447
gdt      564
Name: count, dtype: int64


In [13]:
pbmc_coarse_train_df.to_csv("results/gdt_t_train_df.csv", index=True, header=True)
pbmc_coarse_test_df.to_csv("results/gdt_t_test_df.csv", index=True, header=True)

In [ ]:
import pandas as pd  

# 创建一个空的训练集DataFrame  
pbmc_coarse_train_df = pd.DataFrame()  

# 对每种细胞类型取样150个  
for cell_type in pbmc_pvg_df["coarse_anno"].unique():  
    # 获取当前细胞类型的所有数据  
    cell_df = pbmc_pvg_df[pbmc_pvg_df["coarse_anno"] == cell_type]  
    
    # 对于每种类型，取150个样本或全部（如果少于150个）  
    sample_size = min(150, len(cell_df))  
    sampled_df = cell_df.sample(n=sample_size, random_state=42)  
    
    # 将样本添加到训练集  
    pbmc_coarse_train_df = pd.concat([pbmc_coarse_train_df, sampled_df])  

# 剩余的样本作为测试集  
pbmc_coarse_test_df = pbmc_pvg_df.loc[~pbmc_pvg_df.index.isin(pbmc_coarse_train_df.index)]  

# 验证结果  
print("训练集中每种细胞类型的数量:")  
print(pbmc_coarse_train_df.coarse_anno.value_counts())  
print("\n测试集中每种细胞类型的数量:")  
print(pbmc_coarse_test_df.coarse_anno.value_counts())

In [ ]:
pbmc_coarse_train_df.to_csv("results/pbmc_pvg_coarse_train_df.csv", index=True, header=True)
pbmc_coarse_test_df.to_csv("results/pbmc_pvg_coarse_test_df.csv", index=True, header=True)